# locations_mapped — nightly rebuild (oxjob #762 / #765)

`locations_mapped` is a nightly `CREATE OR REPLACE`: `locations_w_types` JOIN the
`location_work_ids` identity registry (maintained by the `Map_Work_Ids` task,
oxjob #764). The frozen `locations_stale` sidecar was UNIONed here from the 2026-08-12
cutover until 2026-09-11 (oxjob #765: drained to 2.39M rows, the leftover archived as
`locations_stale_leftover_20260911`, then detached). `location_enrichments`
overlays dormant-writer columns works_base still consumes (its language column has a nightly
writer again since oxjob #1871: the cells after it). `locations_mapped_hash`
recreates the MERGE-era `openalex_updated_dt` semantics: stamps carry when content
is unchanged, bump when it is not. Acquisition `endpoint_id` is carried from the
live Location row. Design + evidence: oxjobs #762 PLAN-v2, #765.

In [0]:
CREATE TABLE IF NOT EXISTS openalex.works.location_enrichments (
  provenance STRING,
  native_id_namespace STRING,
  native_id STRING,
  language_classification STRUCT<language: STRING, score: DOUBLE>,
  referenced_works_count INT,
  referenced_works ARRAY<BIGINT>,
  snapshot_dt DATE
)
CLUSTER BY (provenance, native_id_namespace, native_id)

## Language of new locations (oxjob #1871)
`language_classification` had no writer after the 2025 snapshot: Sep-Oct 2025 a one-off `UPDATE` ran fastText `lid.176.ftz`
over every location, and nothing labelled the locations that arrived after it, so 22.3M works (29% of 2026 articles)
serve `language: null`. This step labels each location the night it first appears, before the rebuild below reads
`location_enrichments`: same model file and recipe as the snapshot (title + first 1,000 characters of the abstract +
source name, lowercased, top label; agrees with the snapshot's label on 2,976 of 3,000 random locations). It fills gaps only: a location whose record
declares a language is not labelled, so a label never overrules a declaration (on 26 Oct, 12,863 of 240,672 new labels
would have; declared vs detected is oxjob #1249's question). Locations with no title and no abstract stay unlabelled, and a label under 0.3 confidence counts as no language
(mostly image URLs, species names and file names, which have none). Older unlabelled locations were labelled once by the oxjob #1871 backfill
(Jason's guardrail yes); this step catches anything left. The model ships inside a wheel on the volume
because UC Python UDFs cannot read volumes; `fasttext-predict` (fastText's own C++ predict, no numpy) is a wheel
on the same volume (the UDF sandbox is aarch64 / Python 3.12), so nothing installs from the network during the run. A failed
prediction returns NULL rather than failing the night. Revert: delete these two cells (rows already written stay and are harmless).

In [0]:
CREATE OR REPLACE FUNCTION openalex.common.classify_language_lid176(s STRING)
RETURNS STRUCT<language: STRING, score: DOUBLE>
LANGUAGE PYTHON
DETERMINISTIC
COMMENT 'oxjob #1871: fastText lid.176.ftz language id (the 2025 language_classification model). Lowercases, newlines to spaces, top-1 label; NULL for empty text or any error. Wheel: /Volumes/openalex/works/models/fasttext/'
ENVIRONMENT (dependencies = '["/Volumes/openalex/works/models/fasttext/fasttext_predict-0.9.2.4-cp312-cp312-manylinux_2_17_aarch64.manylinux2014_aarch64.whl", "/Volumes/openalex/works/models/fasttext/openalex_lid176-1.0.0-py3-none-any.whl"]', environment_version = 'None')
AS $$
try:
    from openalex_lid176 import classify
    return classify(s)
except Exception:
    return None
$$

In [0]:
-- oxjob #1871: label every location that has text, declares no language and has no label yet: the night's new
-- locations, plus any older one the backfill (oxjob #1871, after Jason's guardrail yes) missed.
-- Fill only: a location whose record declares a language keeps it (no label written), so the label never overrules
-- a publisher's, repository's or NLM's declaration (declared vs detected is oxjob #1249's open question).
-- Insert-only in practice: a new location has no enrichment row; the MATCHED branch only fills a NULL label, so a
-- re-run never changes a label. Keys stay unique (one source row per key), so the rebuild's LEFT JOIN cannot fan out.
-- The cap bounds a night that brings an unusual flood of new locations; the rest are labelled the next night.
MERGE INTO openalex.works.location_enrichments e
USING (
  -- A label under 0.3 confidence is stored as no language: on 300 + 300 random fills, Opus found no language to read
  -- in 88 + 78 of those below it (herbarium image URLs, species names, file names) and a right label in 1 + 6. The
  -- location still gets a row, so it is not re-read every night.
  SELECT provenance, native_id_namespace, native_id,
         CASE WHEN lc IS NULL THEN NULL
              WHEN lc.score >= 0.3 THEN lc
              ELSE named_struct('language', CAST(NULL AS STRING), 'score', lc.score) END AS language_classification
  FROM (
  SELECT c.provenance, c.native_id_namespace, c.native_id,
         openalex.common.classify_language_lid176(concat_ws(' ', c.title, substr(c.abstract, 1, 1000), c.source_name)) AS lc
  FROM (
    SELECT t.provenance, t.native_id_namespace, t.native_id, t.title, t.abstract, t.source_name
    FROM openalex.works.locations_w_types t
    JOIN openalex.works.location_work_ids r
      ON  t.provenance = r.provenance
      AND t.native_id_namespace = r.native_id_namespace
      AND t.native_id = r.native_id
    LEFT ANTI JOIN (
      SELECT provenance, native_id_namespace, native_id
      FROM openalex.works.location_enrichments
      WHERE language_classification IS NOT NULL
    ) x
      ON  t.provenance = x.provenance
      AND t.native_id_namespace = x.native_id_namespace
      AND t.native_id = x.native_id
    WHERE t.language IS NULL
      AND (coalesce(length(trim(t.title)), 0) > 0 OR coalesce(length(trim(t.abstract)), 0) > 0)
    QUALIFY ROW_NUMBER() OVER (
      PARTITION BY t.provenance, t.native_id_namespace, t.native_id
      ORDER BY t.updated_date DESC) = 1
    LIMIT 10000000
  ) c
  )
) n
ON  e.provenance = n.provenance
AND e.native_id_namespace = n.native_id_namespace
AND e.native_id = n.native_id
WHEN MATCHED AND e.language_classification IS NULL AND n.language_classification IS NOT NULL THEN
  UPDATE SET e.language_classification = n.language_classification
WHEN NOT MATCHED AND n.language_classification IS NOT NULL THEN
  INSERT (provenance, native_id_namespace, native_id, language_classification, referenced_works_count, referenced_works, snapshot_dt)
  VALUES (n.provenance, n.native_id_namespace, n.native_id, n.language_classification, NULL, NULL, current_date())

## Hash baseline (first run only) — change detection for `openalex_updated_dt`
Payload excludes the two bookkeeping stamps. This bootstrap intentionally retains
the pre-lineage formula. The rebuild hash adds `endpoint_id` only when it is non-NULL,
so the cutover bumps rows gaining truthful acquisition lineage without a full-table
rebaseline; NULL-lineage and frozen stale rows retain their prior hash.

In [0]:
CREATE TABLE IF NOT EXISTS openalex.works.locations_mapped_hash
CLUSTER BY (provenance, native_id_namespace, native_id)
AS
SELECT provenance, native_id_namespace, native_id, work_id,
       xxhash64(to_json(struct(
         work_id, work_id_source, merge_key, key_lineage, title, normalized_title,
         authors, ids, raw_type, type, version, license, language, language_classification,
         published_date, created_date, updated_date, issue, volume, first_page, last_page,
         is_retracted, abstract, source_name, publisher, funders, references, urls,
         pdf_url, landing_page_url, pdf_s3_id, grobid_s3_id, mesh, is_oa, is_oa_source,
         referenced_works_count, referenced_works, abstract_inverted_index,
         authors_exist, affiliations_exist, is_corresponding_exists, best_doi, source_id
       ))) AS payload_hash,
       openalex_updated_dt
FROM openalex.works.locations_mapped
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY provenance, native_id_namespace, native_id, work_id
  ORDER BY openalex_updated_dt DESC NULLS LAST) = 1

## Mojibake repair UDFs (oxjob #801)
Publishers deposit double-encoded UTF-8 ("UniversitÃ©") — ~179K works, crossref-dominant.
The rebuild below repairs affiliation-string mojibake at map time for every provenance;
changed payloads bump `openalex_updated_dt` via the hash mechanism, which propagates the
repair downstream (works_base → work_authors → authorships → ES) with no separate sweep.
Two functions: `repair_mojibake` (scalar; also used by SyncRasCurations to re-key curations)
and `repair_mojibake_names_json` (batch over a JSON array of a row's distinct gate-matching
names, returning only the changed pairs — Spark cannot call an external UDF inside
`transform()` lambdas, and whole-authors-array payloads OOMed the warehouse Python sandbox
on hyperauthorship works, so the rebuild joins the pairs back as a map). Both are conservative by
construction (whole-string strict round-trip + restricted lead signature; 0 false positives
on a 3K clean-string corpus): unrepairable text passes through unchanged. Source of truth +
unit tests: oxjobs #801 `work/mojibake_repair.py`.


In [ ]:
CREATE OR REPLACE FUNCTION openalex.common.repair_mojibake(s STRING)
RETURNS STRING
LANGUAGE PYTHON
DETERMINISTIC
COMMENT 'oxjob #801: repairs UTF-8-read-as-latin1/cp1252/latin2/cp1250 mojibake in text (e.g. UniversitÃ© -> Université); returns input unchanged unless confidently repairable. Source of truth + tests: oxjobs #801 work/mojibake_repair.py'
AS $$

import unicodedata

MAX_ROUNDS = 4  # corpus has strings mojibaked up to ~5 levels deep -- each round strictly shrinks

# --- unified latin-1 + cp1252 reverse map ------------------------------------
# chars <= U+00FF map to their own codepoint (latin-1). cp1252's specials
# (which all sit > U+00FF) map to the 0x80-0x9F byte cp1252 decoded them from.
_CP1252_REV = {}
for _b in range(0x80, 0xA0):
    try:
        _CP1252_REV[bytes([_b]).decode("cp1252")] = _b
    except UnicodeDecodeError:
        pass  # 0x81 0x8D 0x8F 0x90 0x9D are undefined in cp1252

# --- detection ----------------------------------------------------------------
# Core leads: chars whose latin-1 byte is a UTF-8 lead byte for scripts that
# actually occur in affiliations. C3=Ã (Latin-1 sup), C2=Â (Latin-1 punct),
# C4=Ä C5=Å (Latin Ext-A), CE=Î CF=Ï (Greek), D0=Ð D1=Ñ (Cyrillic),
# E2=â (general punctuation, 3-byte).
_CORE_LEADS_2 = set("ÃÂÄÅÌÎÏÐÑ")  # Ì (CC) covers NFD combining-mark mojibake ("MuÌ\u0088nster")
# Peripheral 2-byte leads: allowed to *participate* in repair but not sufficient
# alone to trigger it (kills the Ö/Ü + NBSP legit-typography corruption risk).
_PERIPHERAL_LEADS_2 = set(
    "ÆÇÒÓÔÕÖØÙÚÛÜ"
)



def _is_cont(ch):
    """Char maps (via unified map) to a UTF-8 continuation byte 0x80-0xBF?"""
    cp = ord(ch)
    if 0x80 <= cp <= 0xBF:
        return True
    return _CP1252_REV.get(ch, 0) >= 0x80  # cp1252 specials are all 0x80-0x9F




def detect(s):
    """True if s contains at least one CORE mojibake sequence (latin-1/cp1252 read)."""
    if not s:
        return False
    n = len(s)
    for i, ch in enumerate(s):
        if ch in _CORE_LEADS_2 and i + 1 < n and _is_cont(s[i + 1]):
            return True
        if ch == "â" and i + 2 < n and _is_cont(s[i + 1]) and _is_cont(s[i + 2]):
            return True
        cp = ord(ch)
        # 3-byte leads à-á ã-ï (skip â handled above) need two continuations,
        # at least one of which must not be NBSP: legit "Umeå\xa0\xa0University"
        # (å + typographic NBSPs) is byte-wise valid UTF-8 for a rare CJK char
        # and would otherwise corrupt.
        if (0xE0 <= cp <= 0xEF and cp != 0xE2 and i + 2 < n
                and _is_cont(s[i + 1]) and _is_cont(s[i + 2])
                and not (s[i + 1] == "\xa0" and s[i + 2] == "\xa0")):
            return True
        # 4-byte leads ð-ô need three continuations (same NBSP rule)
        if (0xF0 <= cp <= 0xF4 and i + 3 < n
                and _is_cont(s[i + 1]) and _is_cont(s[i + 2]) and _is_cont(s[i + 3])
                and not (s[i + 1] == s[i + 2] == s[i + 3] == "\xa0")):
            return True
    return False


def _fallback_signature(s, codec):
    """True if s contains a DISTINCTIVE lead+continuation pair under the given
    single-byte codec (ISO-8859-2 / cp1250 reads: UTF-8 lead bytes C3/C4/C5
    decode to Ă/Ä/Ĺ there). "Distinctive" = the lead char maps to a different
    byte than its own codepoint, i.e. only that codec's read produces it —
    without this, latin2's É (0xC9, same as latin-1) would reopen the
    "UNIVERSITÉ\\xa0PARIS" corruption trap the core detector closed."""
    if not s:
        return False
    def byte_of(ch):
        try:
            return ch.encode(codec)[0]
        except (UnicodeEncodeError, IndexError):
            return None
    n = len(s)
    for i, ch in enumerate(s):
        b = byte_of(ch)
        if (b is not None and 0xC2 <= b <= 0xDF and b != ord(ch) and i + 1 < n):
            b2 = byte_of(s[i + 1])
            if b2 is not None and 0x80 <= b2 <= 0xBF:
                return True
    return False


# --- repair -------------------------------------------------------------------

def _to_bytes_unified(s):
    out = bytearray()
    for ch in s:
        cp = ord(ch)
        if cp <= 0xFF:
            out.append(cp)
        else:
            b = _CP1252_REV.get(ch)
            if b is None:
                return None
            out.append(b)
    return bytes(out)




def _validate(s, raw):
    if raw is None:
        return None
    try:
        fixed = raw.decode("utf-8", errors="strict")
    except UnicodeDecodeError:
        return None
    if len(fixed) >= len(s) or "\ufffd" in fixed:
        return None
    return fixed


def _badness(s):
    """Count of chars implausible in repaired text (symbols, controls). Used to
    pick among codec candidates: a wrong-codec 'repair' that still validates
    leaves stray symbols (e.g. cp1250 text pushed through latin2 yields '÷')."""
    score = 0
    for ch in s:
        if ch in "\n\r\t":
            continue
        cat = unicodedata.category(ch)
        if cat.startswith("C"):
            score += 3
        elif cat.startswith("S"):
            score += 1
    return score


def _repair_once(s):
    """One round of repair. Returns repaired string, or None if not repairable.

    All candidate codecs are tried and the least-bad validated result wins —
    ISO-8859-2 and cp1250 reads produce overlapping signatures, and the wrong
    codec can still yield valid-UTF-8 garbage, so first-match is not safe.
    """
    candidates = []
    if detect(s):
        fixed = _validate(s, _to_bytes_unified(s))
        if fixed is not None:
            candidates.append(fixed)
    # fallback single-byte codecs (Central European reads): the codec's own
    # encoder IS the reverse map -- strict encode fails on any unmappable char.
    for codec in ("iso-8859-2", "cp1250"):
        if _fallback_signature(s, codec):
            try:
                raw = s.encode(codec, errors="strict")
            except UnicodeEncodeError:
                continue
            fixed = _validate(s, raw)
            if fixed is not None:
                candidates.append(fixed)
    if not candidates:
        return None
    return min(candidates, key=_badness)  # ties keep candidate order (min is stable)


def _fix_bare_c1(s):
    """Map surviving C1 controls (U+0080-U+009F) to their cp1252 chars."""
    if not any(0x80 <= ord(ch) <= 0x9F for ch in s):
        return s
    out = []
    for ch in s:
        if 0x80 <= ord(ch) <= 0x9F:
            try:
                ch = bytes([ord(ch)]).decode("cp1252")
            except UnicodeDecodeError:
                pass  # 0x81 0x8D 0x8F 0x90 0x9D undefined -> keep
        out.append(ch)
    return "".join(out)


def repair(s):
    """Repair mojibake in s -- returns s unchanged if not confidently repairable."""
    if not s:
        return s
    cur = s
    for _ in range(MAX_ROUNDS):
        nxt = _repair_once(cur)
        if nxt is None:
            break
        cur = nxt
    if cur is not s:
        cur = _fix_bare_c1(cur)
    return cur

return repair(s)
$$

In [ ]:
CREATE OR REPLACE FUNCTION openalex.common.repair_mojibake_names_json(names_json STRING)
RETURNS STRING
LANGUAGE PYTHON
DETERMINISTIC
COMMENT 'oxjob #801: batch-repairs mojibake over a JSON array of affiliation-name strings; returns a JSON array of {o, r} pairs for the strings it changed. Replaces repair_mojibake_authors_json: whole-authors-array payloads OOMed the warehouse Python sandbox on hyperauthorship works, so callers pass only the row''s distinct gate-matching names. Same repair core as openalex.common.repair_mojibake.'
AS $$
import json as _json


import unicodedata

MAX_ROUNDS = 4  # corpus has strings mojibaked up to ~5 levels deep -- each round strictly shrinks

# --- unified latin-1 + cp1252 reverse map ------------------------------------
# chars <= U+00FF map to their own codepoint (latin-1). cp1252's specials
# (which all sit > U+00FF) map to the 0x80-0x9F byte cp1252 decoded them from.
_CP1252_REV = {}
for _b in range(0x80, 0xA0):
    try:
        _CP1252_REV[bytes([_b]).decode("cp1252")] = _b
    except UnicodeDecodeError:
        pass  # 0x81 0x8D 0x8F 0x90 0x9D are undefined in cp1252

# --- detection ----------------------------------------------------------------
# Core leads: chars whose latin-1 byte is a UTF-8 lead byte for scripts that
# actually occur in affiliations. C3=Ã (Latin-1 sup), C2=Â (Latin-1 punct),
# C4=Ä C5=Å (Latin Ext-A), CE=Î CF=Ï (Greek), D0=Ð D1=Ñ (Cyrillic),
# E2=â (general punctuation, 3-byte).
_CORE_LEADS_2 = set("ÃÂÄÅÌÎÏÐÑ")  # Ì (CC) covers NFD combining-mark mojibake ("MuÌ\u0088nster")
# Peripheral 2-byte leads: allowed to *participate* in repair but not sufficient
# alone to trigger it (kills the Ö/Ü + NBSP legit-typography corruption risk).
_PERIPHERAL_LEADS_2 = set(
    "ÆÇÒÓÔÕÖØÙÚÛÜ"
)



def _is_cont(ch):
    """Char maps (via unified map) to a UTF-8 continuation byte 0x80-0xBF?"""
    cp = ord(ch)
    if 0x80 <= cp <= 0xBF:
        return True
    return _CP1252_REV.get(ch, 0) >= 0x80  # cp1252 specials are all 0x80-0x9F




def detect(s):
    """True if s contains at least one CORE mojibake sequence (latin-1/cp1252 read)."""
    if not s:
        return False
    n = len(s)
    for i, ch in enumerate(s):
        if ch in _CORE_LEADS_2 and i + 1 < n and _is_cont(s[i + 1]):
            return True
        if ch == "â" and i + 2 < n and _is_cont(s[i + 1]) and _is_cont(s[i + 2]):
            return True
        cp = ord(ch)
        # 3-byte leads à-á ã-ï (skip â handled above) need two continuations,
        # at least one of which must not be NBSP: legit "Umeå\xa0\xa0University"
        # (å + typographic NBSPs) is byte-wise valid UTF-8 for a rare CJK char
        # and would otherwise corrupt.
        if (0xE0 <= cp <= 0xEF and cp != 0xE2 and i + 2 < n
                and _is_cont(s[i + 1]) and _is_cont(s[i + 2])
                and not (s[i + 1] == "\xa0" and s[i + 2] == "\xa0")):
            return True
        # 4-byte leads ð-ô need three continuations (same NBSP rule)
        if (0xF0 <= cp <= 0xF4 and i + 3 < n
                and _is_cont(s[i + 1]) and _is_cont(s[i + 2]) and _is_cont(s[i + 3])
                and not (s[i + 1] == s[i + 2] == s[i + 3] == "\xa0")):
            return True
    return False


def _fallback_signature(s, codec):
    """True if s contains a DISTINCTIVE lead+continuation pair under the given
    single-byte codec (ISO-8859-2 / cp1250 reads: UTF-8 lead bytes C3/C4/C5
    decode to Ă/Ä/Ĺ there). "Distinctive" = the lead char maps to a different
    byte than its own codepoint, i.e. only that codec's read produces it —
    without this, latin2's É (0xC9, same as latin-1) would reopen the
    "UNIVERSITÉ\\xa0PARIS" corruption trap the core detector closed."""
    if not s:
        return False
    def byte_of(ch):
        try:
            return ch.encode(codec)[0]
        except (UnicodeEncodeError, IndexError):
            return None
    n = len(s)
    for i, ch in enumerate(s):
        b = byte_of(ch)
        if (b is not None and 0xC2 <= b <= 0xDF and b != ord(ch) and i + 1 < n):
            b2 = byte_of(s[i + 1])
            if b2 is not None and 0x80 <= b2 <= 0xBF:
                return True
    return False


# --- repair -------------------------------------------------------------------

def _to_bytes_unified(s):
    out = bytearray()
    for ch in s:
        cp = ord(ch)
        if cp <= 0xFF:
            out.append(cp)
        else:
            b = _CP1252_REV.get(ch)
            if b is None:
                return None
            out.append(b)
    return bytes(out)




def _validate(s, raw):
    if raw is None:
        return None
    try:
        fixed = raw.decode("utf-8", errors="strict")
    except UnicodeDecodeError:
        return None
    if len(fixed) >= len(s) or "\ufffd" in fixed:
        return None
    return fixed


def _badness(s):
    """Count of chars implausible in repaired text (symbols, controls). Used to
    pick among codec candidates: a wrong-codec 'repair' that still validates
    leaves stray symbols (e.g. cp1250 text pushed through latin2 yields '÷')."""
    score = 0
    for ch in s:
        if ch in "\n\r\t":
            continue
        cat = unicodedata.category(ch)
        if cat.startswith("C"):
            score += 3
        elif cat.startswith("S"):
            score += 1
    return score


def _repair_once(s):
    """One round of repair. Returns repaired string, or None if not repairable.

    All candidate codecs are tried and the least-bad validated result wins —
    ISO-8859-2 and cp1250 reads produce overlapping signatures, and the wrong
    codec can still yield valid-UTF-8 garbage, so first-match is not safe.
    """
    candidates = []
    if detect(s):
        fixed = _validate(s, _to_bytes_unified(s))
        if fixed is not None:
            candidates.append(fixed)
    # fallback single-byte codecs (Central European reads): the codec's own
    # encoder IS the reverse map -- strict encode fails on any unmappable char.
    for codec in ("iso-8859-2", "cp1250"):
        if _fallback_signature(s, codec):
            try:
                raw = s.encode(codec, errors="strict")
            except UnicodeEncodeError:
                continue
            fixed = _validate(s, raw)
            if fixed is not None:
                candidates.append(fixed)
    if not candidates:
        return None
    return min(candidates, key=_badness)  # ties keep candidate order (min is stable)


def _fix_bare_c1(s):
    """Map surviving C1 controls (U+0080-U+009F) to their cp1252 chars."""
    if not any(0x80 <= ord(ch) <= 0x9F for ch in s):
        return s
    out = []
    for ch in s:
        if 0x80 <= ord(ch) <= 0x9F:
            try:
                ch = bytes([ord(ch)]).decode("cp1252")
            except UnicodeDecodeError:
                pass  # 0x81 0x8D 0x8F 0x90 0x9D undefined -> keep
        out.append(ch)
    return "".join(out)


def repair(s):
    """Repair mojibake in s -- returns s unchanged if not confidently repairable."""
    if not s:
        return s
    cur = s
    for _ in range(MAX_ROUNDS):
        nxt = _repair_once(cur)
        if nxt is None:
            break
        cur = nxt
    if cur is not s:
        cur = _fix_bare_c1(cur)
    return cur


if names_json is None:
    return None
try:
    names = _json.loads(names_json)
except Exception:
    return None
out = []
for n in names or []:
    if isinstance(n, str):
        r = repair(n)
        if r != n:
            out.append({"o": n, "r": r})
return _json.dumps(out, ensure_ascii=False)
$$

In [ ]:
DROP FUNCTION IF EXISTS openalex.common.repair_mojibake_authors_json

## HTML-entity decode UDFs (oxjob #2107)
Crossref deposits (IEEE above all) carry affiliation text HTML-escaped: `Technische Universit&#x00E4;t Berlin`,
`Xi&#x0027;an`, `Policy &amp; Management`, ~1.0M works. The rebuild below decodes them at map time for every
provenance, BEFORE #801's mojibake repair (a decoded reference can reveal mojibake). Rules (strict, because these
strings are exact-match keys): only well-formed references ending in a semicolon; never to `<` or `>` (that would make
markup, and CreateWorksBase drops strings matching `<(div|span|...)`), never to a control character; whitespace results
become one space; in a string that already carries #801's mojibake signature only ASCII results decode, the rest wait for
#801's repair (whole-string: a decoded en dash made it refuse, night 11-10). The gates are written with `\x3B`, never a literal semicolon (the warehouse statement splitter).
Institution answers and curations follow the decoded keys: lookup + matcher answers seeded once by #2107
(`openalex.transient.oxjob2107_seed_*`), curations re-keyed nightly in SyncRasCurations, the frozen legacy side decoded
in CreateWorksBase. Source of truth + tests: oxjobs #2107 `work/decode_entities.py`.
Revert: the `entity_rows` / `entity_maps` / `decoded` CTEs out (gate_rows and repaired read `unioned` again).

In [ ]:
CREATE OR REPLACE FUNCTION openalex.common.decode_affiliation_entities(s STRING)
RETURNS STRING
LANGUAGE PYTHON
DETERMINISTIC
COMMENT 'oxjob #2107: decodes HTML character references in affiliation text (Technische Universit &#x00E4 t becomes Technische Universität). Only well-formed references ending in a semicolon, never to < or > or a control character, whitespace results become one space, input returned unchanged when nothing decodes, only ASCII results decoded in a string that already carries mojibake (left for the #801 repair). Source of truth + tests: oxjobs #2107 work/decode_entities.py'
AS $$

import html as _html
import re as _re
from html.entities import html5 as _HTML5

_SEMI = chr(59)
_REF = _re.compile("&(#[0-9]{1,7}|#[xX][0-9a-fA-F]{1,6}|[A-Za-z][A-Za-z0-9]{1,31})" + _SEMI)
_SPACES = _re.compile("[ \\t\\r\\n\\f\\v\\u0085\\u00a0\\u1680\\u2000-\\u200a\\u2028\\u2029\\u202f\\u205f\\u3000]+")
_KEEP = ("<", ">")
# In a string already carrying #801's mojibake signature (gate_rows' regex) only references that decode to ASCII
# are decoded (&amp -> &), the rest wait: #801's repair is whole-string, and a decoded non-ASCII character (an en
# dash from &ndash) made it refuse (oxjob #2107, night 11-10: 451 strings lost their repaired accents).
_MOJI = _re.compile("[\\u00C2-\\u00DF\\u00E0-\\u00EF][\\u0080-\\u00BF\\u20AC\\u201A\\u0192\\u201E\\u2026\\u2020\\u2021\\u02C6\\u2030\\u0160\\u2039\\u0152\\u017D\\u2018\\u2019\\u201C\\u201D\\u2022\\u2013\\u2014\\u02DC\\u2122\\u0161\\u203A\\u0153\\u017E\\u0178]|[\\u0102\\u0139]")


def _ok(ch):
    if not ch or ch in _KEEP:
        return False
    for c in ch:
        cp = ord(c)
        if cp == 0xFFFD or 0xD800 <= cp <= 0xDFFF or (cp & 0xFFFE) == 0xFFFE or 0xFDD0 <= cp <= 0xFDEF:
            return False
        if (cp < 0x20 or 0x7F <= cp <= 0x9F) and c not in "\t\n\r\f\v\x85":
            return False
    return True


def _sub(m):
    tok = m.group(0)
    body = m.group(1)
    if body[0] == "#":
        ch = _html.unescape(tok)
        if ch == tok:
            return tok
    else:
        ch = _HTML5.get(body + _SEMI)
        if ch is None:
            return tok
    return ch if _ok(ch) else tok


def _sub_ascii(m):
    ch = _sub(m)
    return ch if ch.isascii() else m.group(0)


def decode(s):
    if not s or "&" not in s:
        return s
    sub = _sub_ascii if _MOJI.search(s) else _sub
    cur = s
    for _ in range(3):
        nxt = _REF.sub(sub, cur)
        if nxt == cur:
            break
        cur = nxt
    if cur == s:
        return s
    cur = _SPACES.sub(" ", cur).strip()
    return cur if cur else s

return decode(s)
$$

In [ ]:
CREATE OR REPLACE FUNCTION openalex.common.decode_affiliation_entities_names_json(names_json STRING)
RETURNS STRING
LANGUAGE PYTHON
DETERMINISTIC
COMMENT 'oxjob #2107: batch form of openalex.common.decode_affiliation_entities over a JSON array of affiliation-name strings, returns a JSON array of {o, r} pairs for the strings it changed (same calling pattern as repair_mojibake_names_json, oxjob #801).'
AS $$
import json as _json

import html as _html
import re as _re
from html.entities import html5 as _HTML5

_SEMI = chr(59)
_REF = _re.compile("&(#[0-9]{1,7}|#[xX][0-9a-fA-F]{1,6}|[A-Za-z][A-Za-z0-9]{1,31})" + _SEMI)
_SPACES = _re.compile("[ \\t\\r\\n\\f\\v\\u0085\\u00a0\\u1680\\u2000-\\u200a\\u2028\\u2029\\u202f\\u205f\\u3000]+")
_KEEP = ("<", ">")
# In a string already carrying #801's mojibake signature (gate_rows' regex) only references that decode to ASCII
# are decoded (&amp -> &), the rest wait: #801's repair is whole-string, and a decoded non-ASCII character (an en
# dash from &ndash) made it refuse (oxjob #2107, night 11-10: 451 strings lost their repaired accents).
_MOJI = _re.compile("[\\u00C2-\\u00DF\\u00E0-\\u00EF][\\u0080-\\u00BF\\u20AC\\u201A\\u0192\\u201E\\u2026\\u2020\\u2021\\u02C6\\u2030\\u0160\\u2039\\u0152\\u017D\\u2018\\u2019\\u201C\\u201D\\u2022\\u2013\\u2014\\u02DC\\u2122\\u0161\\u203A\\u0153\\u017E\\u0178]|[\\u0102\\u0139]")


def _ok(ch):
    if not ch or ch in _KEEP:
        return False
    for c in ch:
        cp = ord(c)
        if cp == 0xFFFD or 0xD800 <= cp <= 0xDFFF or (cp & 0xFFFE) == 0xFFFE or 0xFDD0 <= cp <= 0xFDEF:
            return False
        if (cp < 0x20 or 0x7F <= cp <= 0x9F) and c not in "\t\n\r\f\v\x85":
            return False
    return True


def _sub(m):
    tok = m.group(0)
    body = m.group(1)
    if body[0] == "#":
        ch = _html.unescape(tok)
        if ch == tok:
            return tok
    else:
        ch = _HTML5.get(body + _SEMI)
        if ch is None:
            return tok
    return ch if _ok(ch) else tok


def _sub_ascii(m):
    ch = _sub(m)
    return ch if ch.isascii() else m.group(0)


def decode(s):
    if not s or "&" not in s:
        return s
    sub = _sub_ascii if _MOJI.search(s) else _sub
    cur = s
    for _ in range(3):
        nxt = _REF.sub(sub, cur)
        if nxt == cur:
            break
        cur = nxt
    if cur == s:
        return s
    cur = _SPACES.sub(" ", cur).strip()
    return cur if cur else s

if names_json is None:
    return None
try:
    names = _json.loads(names_json)
except Exception:
    return None
out = []
for n in names or []:
    if isinstance(n, str):
        r = decode(n)
        if r != n:
            out.append({"o": n, "r": r})
return _json.dumps(out, ensure_ascii=False)
$$

## Elsevier Open Manuscripts: the manuscript's own address (oxjob #1934)
Elsevier registers its Open Manuscript programme in Crossref as an `am` licence
(`elsevier.com/open-access/userlicense`) with a start date, and Crossref.py reads it as an open accepted
manuscript (oxjob #1118). But the location kept only the DOI, so ~297K works served the paywalled article page as
their best open copy, with no PDF link. ScienceDirect hosts each manuscript at
`https://www.sciencedirect.com/science/article/am/pii/<PII>`, which sends a reader straight to the manuscript PDF;
the PII is in the same deposit (`resource.primary.URL`, the linkinghub address; `alternative-id` agrees on every
record). This table lists, from today's Crossref records, every Elsevier DOI whose `am` licence has started; the
rebuild below gives that address as the `pdf_url` of a Crossref location it labels an accepted manuscript and that
has no PDF yet. The landing page stays the DOI (this location is also the work's primary location). Unpaywall served
the same address as `url_for_pdf` (oadoi `pub.py`, Open Manuscript branch). Revert: in the rebuild, `pdf_url` back to
`t.pdf_url` and drop the `om` join; then `DROP TABLE openalex.works.elsevier_open_manuscript_pii`.

In [ ]:
-- oxjob #1934: one row per Elsevier DOI whose Open Manuscript licence has started, with its PII. Same gate as
-- Crossref.py's elsevier_oa_started('am') (oxjob #1118): the DOI still resolves to linkinghub.elsevier.com and the am
-- userlicense start date has passed. PIIs outside the 'S' + 16 form (57 on 10 Nov: 1978-84 journal items, book
-- chapters) are left out. No dollar sign in this SQL: the job substitutes them.
CREATE OR REPLACE TABLE openalex.works.elsevier_open_manuscript_pii
CLUSTER BY (native_id)
AS
SELECT native_id, pii
FROM (
  SELECT native_id, regexp_extract(resource.primary.URL, '/pii/([^/?#]+)', 1) AS pii
  FROM openalex.crossref.crossref_deduplicated
  WHERE resource.primary.URL LIKE '%linkinghub.elsevier.com/%'
    AND exists(license, x -> x.URL LIKE '%elsevier.com/open-access/userlicense%'
                             AND x.`content-version` = 'am'
                             AND to_timestamp(x.start.`date-time`) <= current_timestamp())
)
WHERE pii RLIKE '^S[0-9X]{16}' AND length(pii) = 17
QUALIFY ROW_NUMBER() OVER (PARTITION BY native_id ORDER BY pii) = 1

## Crossref subtitles (oxjob #1814)
Crossref deposits a title's subtitle in its own `subtitle` element, and Crossref.py builds `title` from `title[0]` only, so
~4.2M works served a half title ('Literary Territories' for 'Literary Territories: Cartographical Thinking in Late
Antiquity'; 300K+ ended on a bare ':'). This table holds, for every Crossref record whose first non-blank subtitle is not
already in its location's title, the title with the subtitle joined (`title_out`), or NULL and the guard that skipped it
(`skip_reason`: datasets, 'other' type, codes and numbers, section and presentation labels, citation lines, series lines,
running heads, standfirsts, translations, a title that already carries its own subtitle; rules in the cell).
The rebuild below serves `title_out` for a Crossref location only when its title is exactly the `title_in` judged here.
`normalized_title` and `merge_key` keep the old title, so no work changes identity (#807's rule for display titles).
Bar: fresh 100 judged by Opus 5.5 against the publisher's deposit and landing page, 99/100 right (oxjobs #1814).
Revert: drop the `sub` join and `title_out` in the rebuild, remove this cell, `DROP TABLE openalex.works.crossref_subtitle_titles`.

In [ ]:
-- oxjob #1814: Crossref subtitle -> served title, with guards (source + bar: oxjobs #1814 work/subtitle_guarded.sql).
-- Rebuilt each run from today's crossref_deduplicated and locations_w_types (~6.6M rows, ~1.5 min on a 2X-Large).
CREATE OR REPLACE TABLE openalex.works.crossref_subtitle_titles
CLUSTER BY (native_id)
COMMENT 'oxjob #1814: Crossref records whose subtitle is missing from the location title; title_out = title joined with the subtitle, NULL when a guard skips it (skip_reason). Rebuilt by CreateLocationsMapped each run.'
AS
WITH cr AS (
  SELECT native_id, type AS crtype, publisher, coalesce(`container-title`, array()) AS ct_arr,
         element_at(filter(subtitle, x -> trim(x) <> ''), 1) AS st_raw
  FROM openalex.crossref.crossref_deduplicated
  WHERE size(filter(coalesce(subtitle, array()), x -> trim(x) <> '')) > 0
    AND type NOT IN ('component', 'grant')
),
loc AS (
  SELECT native_id, title
  FROM openalex.works.locations_w_types
  WHERE provenance = 'crossref'
  QUALIFY ROW_NUMBER() OVER (PARTITION BY provenance, native_id_namespace, native_id ORDER BY updated_date DESC) = 1
),
j AS (
  SELECT cr.native_id, cr.crtype, cr.publisher, cr.ct_arr, cr.st_raw, loc.title AS t,
    -- clean the subtitle like the ingest cleans titles: tags out, common entities decoded, mojibake repaired,
    -- whitespace collapsed, leading/trailing dashes, colons and stray punctuation trimmed
    regexp_replace(regexp_replace(trim(regexp_replace(
      openalex.common.repair_mojibake(
        replace(replace(replace(replace(replace(replace(replace(replace(
          regexp_replace(cr.st_raw, '<[^<>]{0,200}>', ''),
          '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', ''''), '&apos;', ''''), '&nbsp;', ' '), '&#160;', ' '), '&amp;', '&')),
      '[\\s\\x{00A0}\\x{2000}-\\x{200B}\\x{202F}\\x{205F}\\x{3000}]+', ' ')), '^[\\s\\-–—－:;,.·]+', ''), '[\\s\\-–—－:;,·]+$', '') AS st
  FROM cr JOIN loc ON loc.native_id = cr.native_id
),
f AS (
  SELECT *,
    length(regexp_replace(coalesce(t, ''), '[^\\p{L}]', '')) AS t_letters,
    length(regexp_replace(st, '[^\\p{L}]', '')) AS s_letters,
    length(regexp_replace(coalesce(t, ''), '[^\\p{IsLatin}]', '')) / greatest(length(regexp_replace(coalesce(t, ''), '[^\\p{L}]', '')), 1) AS t_latin,
    length(regexp_replace(st, '[^\\p{IsLatin}]', '')) / greatest(length(regexp_replace(st, '[^\\p{L}]', '')), 1) AS s_latin,
    length(regexp_replace(st, '[^\\p{L}]', '')) / greatest(length(regexp_replace(st, '\\s', '')), 1) AS s_letter_share,
    size(split(trim(coalesce(t, '')), '\\s+')) AS t_words,
    size(split(st, '\\s+')) AS s_words,
    filter(split(lower(regexp_replace(st, '[^\\p{L}\\p{N}\\s]', ' ')), '\\s+'), w -> length(w) >= 3) AS s_tok,
    split(lower(regexp_replace(coalesce(t, ''), '[^\\p{L}\\p{N}\\s]', ' ')), '\\s+') AS t_tok
  FROM j
),
g AS (
  SELECT *,
    size(s_tok) AS s_ntok,
    size(filter(s_tok, w -> array_contains(t_tok, w))) AS s_ntok_in_t,
    lower(regexp_replace(st, '^[\\(\\[]+|[\\)\\]\\.]+$', '')) AS st_label,
    regexp_replace(lower(st), '[^\\p{L}\\p{N}]', '') AS st_norm
  FROM f
),
pre AS (
  SELECT *,
    CASE
      WHEN crtype = 'dataset' THEN 'dataset'
      WHEN crtype = 'other' THEN 'type_other'
      WHEN t IS NULL OR t NOT RLIKE '[\\p{L}\\p{N}]' THEN 'blank_title'
      WHEN st = '' OR st IS NULL THEN 'blank_subtitle'
      WHEN contains(regexp_replace(lower(t), '[^\\p{L}\\p{N}]', ''), regexp_replace(lower(st), '[^\\p{L}\\p{N}]', '')) THEN 'already_in_title'
      WHEN st RLIKE '[<>]|&#?[a-zA-Z0-9]{2,8};|\\x{FFFD}|\\?\\?\\?' THEN 'markup_residue'
      -- v2 (tuned on samples A and B, 11 Nov): the title already carries its own subtitle, so the subtitle field holds
      -- something else (a session, congress, column or translation)
      WHEN rtrim(t) RLIKE ':\\s+\\S' THEN 'title_has_subtitle'
      WHEN st RLIKE '^[\\(\\[].*[\\)\\]]$' THEN 'parenthetical'
      WHEN s_letters < 3 OR s_letter_share < 0.5 OR size(s_tok) = 0
        OR (st NOT LIKE '% %' AND st RLIKE '[0-9]' AND length(st) <= 15) THEN 'code_or_number'
      WHEN st_label IN ('original article', 'review article', 'research article', 'original research', 'original research article',
          'research paper', 'original paper', 'original contribution', 'original investigation', 'short communication',
          'brief communication', 'brief report', 'short report', 'editorial', 'letter to the editor', 'letter', 'research',
          'review', 'reviews', 'mini review', 'mini-review', 'literature review', 'literatur review', 'case report', 'case reports',
          'case study', 'clinical case', 'perspective', 'commentary', 'opinion', 'abstract', 'abstracts', 'poster', 'oral presentation',
          'conference paper', 'full paper', 'technical note', 'note', 'notes', 'news', 'book review', 'book reviews', 'review essay',
          'call for papers', 'erratum', 'correction', 'corrigendum', 'article', 'articles', 'research note', 'research notes',
          'artículo original', 'artigo original', 'artículo de revisión', 'artigo de revisão', 'artículo de investigación',
          'artigo de pesquisa', 'relato de caso', 'reporte de caso', 'caso clínico', 'penelitian', 'artikel penelitian',
          'artikel review', 'tinjauan pustaka', 'laporan kasus', 'originalarbeit', 'übersichtsarbeit', 'kasuistik', 'fallbericht',
          'article original', 'article de recherche', 'éditorial', 'compte rendu', 'comptes rendus', 'recension', 'rezension',
          'recenzja', 'reseña', 'resenha', 'in memoriam', 'obituary', 'introduction', 'preface', 'foreword', 'conclusion',
          'english', 'english version', 'in english', 'en', 'eng', 'indonesian', 'bahasa indonesia', 'bahasa inggris', 'spanish',
          'español', 'portuguese', 'português', 'french', 'français', 'german', 'deutsch', 'russian', 'italian', 'italiano',
          'arabic', 'chinese', 'japanese', 'turkish', 'türkçe', 'polish', 'polski', 'full text', 'full-text', 'pdf',
          'scientific note', 'technical perspective', 'editorials', 'letters', 'commentaries', 'original articles',
          'research articles', 'short communications', 'brief reports', 'conclusions', 'konklusion', 'fazit', 'schluss',
          'zusammenfassung', 'summary', 'résumé', 'conclusión', 'conclusão', 'conclusione', 'epilogue', 'prologue',
          'einleitung', 'introducción', 'introdução', 'introduzione', 'vorwort', 'préface', 'prefacio')
        OR st RLIKE '(?i)^(appendix|annex|anhang|anexo|appendice|chapter|kapitel|section)\\s+[A-Z0-9IVXivx]{1,5}\\.?$'
        OR st RLIKE '(?i)\\bhighlights$' THEN 'section_label'
      WHEN s_latin >= 0.5 AND st NOT LIKE '% %' THEN 'one_word'
      WHEN st RLIKE '(?i)presenting author|^(oral|poster)( presentation| session)?\\b|^poster\\s*#|^(abstract|poster|paper|session)\\s*(no\\.?|number|#)?\\s*#?\\s*[0-9]|contact hours?\\b|\\bce credits?\\b' THEN 'presentation_label'
      WHEN st RLIKE '(?i)\\b[0-9]+\\s*(pp|p|pages|pgs|ss|seiten|págs?)\\.?(\\W|$)|\\bisbn\\b|[£\\x{24}€¥]\\s?[0-9]|\\bvol\\.?\\s*[0-9]+.*\\b(no|p|pp)\\.?\\s*[0-9]+|^(by|edited by|reviewed by|ed\\. by|hrsg\\. von|herausgegeben von|par|von) \\p{Lu}'
        OR st RLIKE '^[^,:;]{2,80}, (1[5-9]|20)[0-9]{2}\\.?$'
        OR st RLIKE '(?i)\\b(pp|ss|pgs|págs?|p|s)\\.\\s*[0-9]+|\\bdirected by\\b|\\b[0-9]+ min(utes|\\.)|www\\.|https?://|\\bet al\\b|\\b(born|died|geboren|gestorben)\\s*:'
        OR st RLIKE ',\\s*(1[5-9]|20)[0-9]{2}\\.?$'
        OR t RLIKE '(?i)[£\\x{24}€¥]\\s?[0-9]|\\bisbn\\b|\\b[0-9]+\\s*(pp|pages|seiten)\\.?(\\W|$)|\\b(pp|ss)\\.\\s*[0-9]+'
        OR st RLIKE '\\p{Lu}\\.\\s?,.*\\p{Lu}\\.' THEN 'citation_line'
      WHEN st LIKE 'The IUCN Red List of Threatened Species%' OR st RLIKE '^From:\\s'
        OR st RLIKE '(?i)^data extract from|landolt|^chemical shifts and coupling constants' THEN 'series_line'
      WHEN st LIKE '% = %' THEN 'parallel_title'
      WHEN length(st) > 250 OR (st RLIKE '\\.$' AND s_words >= 12) THEN 'standfirst'
      WHEN publisher LIKE 'Royal College of Nursing%' OR native_id LIKE '10.7748/%' THEN 'standfirst'
      -- a sentence (a lower-case finite verb, not a question) of 6+ words is a deck or teaser, not a subtitle
      WHEN s_words >= 6 AND st NOT RLIKE '\\?$'
        AND st RLIKE '\\b(is|are|was|were|will|has|have|had|can|could|would|should|may|might|must|says|writes|looks|takes|makes|shows|reveals|explains|argues|joins|holds|meet|meets|è|sono|ist|sind|wird|werden|wurde|wurden|est|sont|fue|está|están|foi|são)\\b'
        AND st NOT RLIKE '(?i)^(how|why|what|when|where|who|whether|is|are|can|does|do|should|was|were|will)\\b' THEN 'standfirst'
      WHEN s_words >= 6 AND st LIKE '%;%' THEN 'standfirst'
      WHEN s_ntok >= 2 AND s_ntok_in_t / s_ntok >= 0.5 THEN 'running_head'
      WHEN publisher LIKE 'Wiley%' AND st = upper(st) AND t <> upper(t) AND st RLIKE '\\p{Lu}{3}' THEN 'running_head'
      WHEN t_letters >= 3 AND s_letters >= 3 AND ((t_latin >= 0.5) <> (s_latin >= 0.5)) THEN 'other_script'
    END AS pre_reason
  FROM g
),
lid AS (
  -- language ids only where no cheaper guard already decided
  SELECT p.*,
    CASE WHEN pre_reason IS NULL THEN openalex.common.classify_language_lid176(lower(t)) END AS t_lid,
    CASE WHEN pre_reason IS NULL THEN openalex.common.classify_language_lid176(lower(st)) END AS s_lid
  FROM pre p
)
SELECT native_id, crtype, publisher, t AS title_in, st_raw, st AS subtitle,
  COALESCE(pre_reason,
    CASE WHEN t_lid.language <> s_lid.language AND s_words / t_words BETWEEN 0.5 AND 2.0
              AND ((t_lid.score >= 0.5 AND s_lid.score >= 0.5 AND t_words >= 3 AND s_words >= 3)
                OR (t_lid.score >= 0.3 AND s_lid.score >= 0.3 AND t_words >= 5 AND s_words >= 5)
                OR (s_words / t_words BETWEEN 0.6 AND 1.6 AND s_ntok > 0 AND s_ntok_in_t / s_ntok >= 0.2)) THEN 'translation' END) AS skip_reason,
  CASE WHEN COALESCE(pre_reason,
    CASE WHEN t_lid.language <> s_lid.language AND s_words / t_words BETWEEN 0.5 AND 2.0
              AND ((t_lid.score >= 0.5 AND s_lid.score >= 0.5 AND t_words >= 3 AND s_words >= 3)
                OR (t_lid.score >= 0.3 AND s_lid.score >= 0.3 AND t_words >= 5 AND s_words >= 5)
                OR (s_words / t_words BETWEEN 0.6 AND 1.6 AND s_ntok > 0 AND s_ntok_in_t / s_ntok >= 0.2)) THEN 'translation' END) IS NULL
       THEN substring(concat(rtrim(t), CASE WHEN rtrim(t) RLIKE '[:.?!]$' OR st RLIKE '^[\\(\\[]' THEN ' ' ELSE ': ' END, st), 1, 5000)
  END AS title_out,
  t_lid.language AS t_lang, t_lid.score AS t_score, s_lid.language AS s_lang, s_lid.score AS s_score
FROM lid
-- one row per record, so the rebuild's LEFT JOIN can never fan a location out
QUALIFY ROW_NUMBER() OVER (PARTITION BY native_id ORDER BY title_in, st_raw) = 1


## Rebuild `locations_mapped`
Live side: `locations_w_types` (one row per anchor) JOIN registry JOIN enrichment
overlay, including acquisition `endpoint_id`. The `locations_stale` sidecar is no longer
unioned (detached 2026-09-11, oxjob #765). Endpoint lineage participates in the payload hash only when
non-NULL. `ingested_at` (oxjob #911) is carried from the live row (S3 file arrival for repo/irdb) and stays OUT of the payload hash so restamps never bump `openalex_updated_dt`.

In [0]:
CREATE OR REPLACE TABLE openalex.works.locations_mapped
CLUSTER BY (merge_key.doi, merge_key.pmid, merge_key.arxiv, merge_key.title_author)
TBLPROPERTIES (
  'delta.checkpoint.writeStatsAsJson' = 'false',
  'delta.checkpoint.writeStatsAsStruct' = 'true',
  'delta.enableDeletionVectors' = 'true',
  'delta.feature.deletionVectors' = 'supported',
  'delta.feature.rowTracking' = 'supported',
  'delta.feature.v2Checkpoint' = 'supported')
AS
WITH t AS (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY provenance, native_id_namespace, native_id
           ORDER BY updated_date DESC) AS rwcnt
  FROM openalex.works.locations_w_types
  QUALIFY rwcnt = 1
),
live AS (
  SELECT
    r.work_id,
    r.work_id_source,
    t.merge_key,
    CAST(NULL AS STRING) AS key_lineage,
    t.provenance,
    t.native_id,
    t.native_id_namespace,
    COALESCE(sub.title_out, t.title) AS title, -- oxjob #1814: the Crossref subtitle joined (guarded table above)
    t.normalized_title, -- still the deposit's title[0]: matching keys never move with the subtitle
    -- oxjob #2108: an author entry whose name is only the Ovid / Adis 'not available' marker ('&NA;', '&NA; &NA;', and
    -- the escaped '&amp;Na;,' on Zenodo copies) is not an author. Crossref deposits for ~291K Wolters Kluwer and Adis
    -- items (Reactions Weekly / Inpharma Weekly digests, notices, indexes, unsigned pieces) carry it as their only author;
    -- 21 of 21 publisher pages read show no author. Dropped here, so works_base takes the next source's byline (PubMed
    -- for ~4K works) or the work reads author-less. A name with a marker and real text ('&NA; Jelliffe') is kept.
    -- The ILIKE gate keeps the regexes off every other row. Revert: this CASE back to t.authors.
    CASE WHEN EXISTS(t.authors, a -> a.name ILIKE '%na;%')
         THEN FILTER(t.authors, a -> NOT COALESCE(
                a.name ILIKE '%na;%' AND a.name RLIKE '(?i)&(amp;)*na;'
                AND NOT regexp_replace(a.name, '(?i)&(amp;)*na;', '') RLIKE '[\\p{L}\\p{N}]', FALSE))
         ELSE t.authors
    END AS authors,
    t.ids,
    t.raw_type,
    -- oxjob #2139: a HathiTrust catalogue record (University of Michigan feed, S7407064297) is a whole digitised volume,
    -- but its dc:type is only 'text', which the classifier reads as article (2.57M works typed article, 0 book). The
    -- catalogue's own bibliographic level decides instead: a monograph (BK) is a book, a serial's run (SE) is other (as
    -- the type dictionary's 'serial' and 'periodical'). Kept: dissertation, report, standard. One row per catalogue record
    -- in hathitrust_catalogue_types (notebooks/maintenance/BuildHathiTrustCatalogueTypes.sql, from HathiTrust's Hathifile).
    -- Revert: this CASE back to t.type and drop the ht join below (then the table).
    -- oxjob #2148: same for a B3Kat union-catalogue record (Bavarian/Berlin-Brandenburg library networks, OAI endpoint
    -- 5a4c71918fc62d3e4ce): its dc:type 'text' (or 'Zeitschrift', 'Konferenzschrift') reads as article (3.65M works). B3Kat's
    -- MARC leader decides instead: a monograph of text is a book where the classifier says article; a serial is other where
    -- it says article or book. Component parts and every other type are kept. One row per such record in
    -- b3kat_catalogue_types (notebooks/maintenance/BuildB3KatCatalogueTypes.sql, from B3Kat's CC0 open-data dump).
    -- Revert: drop the two b3 WHEN lines and the b3 join below (then the table).
    CASE WHEN ht.catalogue_type IS NOT NULL AND lower(t.raw_type) = 'text'
              AND coalesce(t.type, '') NOT IN ('dissertation', 'report', 'standard')
         THEN ht.catalogue_type
         WHEN b3.catalogue_type = 'book' AND t.type = 'article' THEN 'book'
         WHEN b3.catalogue_type = 'other' AND t.type IN ('article', 'book') THEN 'other'
         ELSE t.type END AS type,
    -- oxjob #2094: a Crossref posted-content record (preprint, working paper, poster, conference abstract, blog post,
    -- supplement) is never an accepted manuscript: Crossref defines posted content as posted but not formally
    -- published, and Unpaywall always served it as submittedVersion (oadoi pub.py make_preprint). Crossref.py labelled
    -- every one acceptedVersion from May 2025 (4.17M locations); this corrects the stored rows every night, curations
    -- still win downstream. Ranking is unchanged: works_base ranks a Crossref record the same whether accepted or
    -- submitted, so no primary location, best OA location, is_oa or oa_status moves; only version and is_accepted.
    -- Revert: this CASE back to t.version.
    CASE WHEN t.provenance = 'crossref' AND t.raw_type = 'posted-content' THEN 'submittedVersion' ELSE t.version END AS version,
    t.license,
    t.language,
    e.language_classification,
    -- oxjob #2091: a DataCite record with no dates[] was dated by its DOI registration (EMSL/PNNL datasets of
    -- 2019-2025 served as 2026); DataCite.py now takes 1 Jan of the record's publicationYear when the registration
    -- falls in a later year. datacite_publication_year_fix (notebooks/maintenance/BuildDataCitePublicationYearFix.sql)
    -- carries that date for the records parsed before the rule, and applies only while the location still holds the
    -- registration date it was built against (a re-parsed or updated record takes its own date).
    -- Revert: this CASE back to t.published_date and drop the fx join below (then the table).
    -- oxjob #2158: a CiteSeerX location (repository endpoint 7e4fec42e1147584aae, a frozen feed) dates no work. Its
    -- dc:dates are record-keeping dates (the datestamp, the day CiteSeerX created the record) plus, sometimes, a bare year
    -- CiteSeerX's extractor read from the PDF. The crawl dates dated 2.34M works (56 of 57 checkable were published in
    -- another year); the bare years were off by more than a year for 13.9% of 151 confirmed works (bar 2%). So
    -- citeseerx_publication_date_fix (notebooks/maintenance/BuildCiteSeerXPublicationDateFix.sql) gives every dated
    -- record NULL (no year beats a doubtful year: Jason, 26 Oct 2026), applied only while the location still holds the
    -- date it was built against.
    -- Revert: drop the cx WHEN line and the cx join below (then the table).
    CASE WHEN fx.fix_old_date = t.published_date THEN fx.fix_new_date
         WHEN cx.fix_old_date = t.published_date THEN cx.fix_new_date
         ELSE t.published_date END AS published_date,
    t.created_date,
    t.updated_date,
    t.issue,
    t.volume,
    t.first_page,
    t.last_page,
    COALESCE(t.is_retracted, FALSE) AS is_retracted,
    t.abstract,
    t.source_name,
    t.publisher,
    t.funders,
    t.references,
    t.urls,
    t.pdf_url,
    t.landing_page_url,
    t.pdf_s3_id,
    t.grobid_s3_id,
    t.mesh,
    COALESCE(t.is_oa, FALSE) AS is_oa,
    COALESCE(t.is_oa_source, FALSE) AS is_oa_source,
    e.referenced_works_count,
    e.referenced_works,
    t.abstract_inverted_index,
    t.authors_exist,
    t.affiliations_exist,
    t.is_corresponding_exists,
    -- oxjob #1903 (R4): a repository record oxjob #1903 moved off the work whose DOI it carries (a chapter, abstract or
    -- other item under its book's, proceedings' or a neighbour's DOI; pin tagged 'oxjob1903_nodoi') does not show that
    -- DOI: the registrant's item keeps it, so it lands on neither the record's location nor the record's own work.
    CASE WHEN r.work_id_source = 'oxjob1903_nodoi' THEN NULL ELSE t.best_doi END AS best_doi,
    t.source_id,
    t.endpoint_id,
    -- oxjob #911: acquisition timestamp (S3 file arrival for repo/irdb, pipeline read
    -- time for other sources pre-cutover; backfill feed = its 2025-01-16 export batch).
    -- Deliberately NOT in the payload hash below: bookkeeping, must not bump
    -- openalex_updated_dt / churn ES when a refresh restamps it.
    t.ingested_at,
    COALESCE(r.openalex_created_dt, current_date()) AS openalex_created_dt
  FROM t
  LEFT JOIN openalex.works.location_work_ids r
    ON  t.provenance = r.provenance
    AND t.native_id_namespace = r.native_id_namespace
    AND t.native_id = r.native_id
  LEFT JOIN openalex.works.location_enrichments e
    ON  t.provenance = e.provenance
    AND t.native_id_namespace = e.native_id_namespace
    AND t.native_id = e.native_id
  LEFT JOIN openalex.works.citeseerx_publication_date_fix cx -- oxjob #2158 (one row per CiteSeerX record whose date changes, ~5.0M)
    ON  t.provenance = 'repo'
    AND t.native_id_namespace = cx.native_id_namespace
    AND t.native_id = cx.native_id
  LEFT JOIN openalex.works.datacite_publication_year_fix fx -- oxjob #2091 (one row per DataCite DOI)
    ON  t.provenance = 'datacite'
    AND t.native_id_namespace = fx.native_id_namespace
    AND t.native_id = fx.native_id
  LEFT JOIN openalex.works.hathitrust_catalogue_types ht -- oxjob #2139 (one row per HathiTrust catalogue record, ~9.7M)
    ON  t.provenance = 'repo'
    AND t.native_id_namespace = ht.native_id_namespace
    AND t.native_id = ht.native_id
  LEFT JOIN openalex.works.b3kat_catalogue_types b3 -- oxjob #2148 (one row per B3Kat monograph or serial record our feed holds)
    ON  t.provenance = 'repo'
    AND t.native_id_namespace = b3.native_id_namespace
    AND t.native_id = b3.native_id

  LEFT JOIN openalex.works.crossref_subtitle_titles sub -- oxjob #1814 (one row per Crossref record with a subtitle, ~6.6M)
    ON  t.provenance = 'crossref'
    AND t.native_id = sub.native_id
    AND t.title = sub.title_in
),
unioned AS (
  -- oxjob #946: drop the locations that make a work enormous. The list is a static
  -- snapshot built by notebooks/maintenance/BuildLocationDenylist.py: an (work, endpoint)
  -- group of >=100 locations is many distinct items collapsed onto one merge key, not one
  -- item deposited in many repositories. Works left with nothing real ledger out through
  -- TrackDeletedWorks.
  SELECT lu.*
  FROM live lu
  LEFT ANTI JOIN openalex.works.location_denylist d
    ON  lu.provenance          = d.provenance
    AND lu.native_id_namespace = d.native_id_namespace
    AND lu.native_id           = d.native_id
  -- oxjob #1872: Crossref keeps a DOI its registrant deleted as a record in a pseudo-journal ("CrossRef Listing of
  -- Deleted DOIs", member 53084; older ones "Deleted DOIs") whose title is the DOI itself; the DOI resolves to
  -- crossref.org/deleted_DOI.html. That record is not a work: ~160K of them were served as articles dated 2000 on
  -- S4210212300. Drop them here: works left with nothing ledger out through TrackDeletedWorks, works with a real
  -- copy (MAG, repository, PubMed) keep it. Tombstones that still carry a real title (~4.3K, old JSTOR / Springer
  -- items) stay: a real paper under a dead DOI is a dedup question, not a deletion.
  WHERE NOT coalesce(
    lu.provenance = 'crossref'
    AND lower(trim(lu.source_name)) IN ('crossref listing of deleted dois', 'deleted dois')
    AND (lu.title IS NULL OR trim(lu.title) = '' OR trim(lu.title) RLIKE '^10\\.[0-9]{4,}/\\S+$'),
    FALSE)
),
-- oxjob #2107: decode HTML entities in affiliation strings (Technische Universit + the hex reference for a-umlaut +
-- t becomes Technische Universität) for ALL provenances, before the mojibake repair below. Same pattern as #801: a cheap
-- RLIKE gate in a WHERE (the gate uses \x3B, never a literal semicolon), the UDF over the row's DISTINCT gate-matching
-- names in a projection above it, the {o, r} pairs joined back as a map. Only affiliations[].name changes.
-- Revert: drop entity_rows / entity_maps / decoded; gate_rows and repaired read unioned again.
entity_rows AS (
  SELECT provenance, native_id_namespace, native_id, work_id, authors
  FROM unioned
  WHERE EXISTS(authors, a -> EXISTS(a.affiliations, f -> f.name RLIKE '&(#[0-9]{1,7}|#[xX][0-9a-fA-F]{1,6}|[A-Za-z][A-Za-z0-9]{1,31})\\x3B'))
),
entity_maps AS (
  SELECT
    provenance,
    native_id_namespace,
    native_id,
    work_id,
    map_from_entries(from_json(
      openalex.common.decode_affiliation_entities_names_json(to_json(
        array_distinct(filter(
          flatten(transform(authors, a -> coalesce(transform(a.affiliations, f -> f.name), array()))),
          n -> n RLIKE '&(#[0-9]{1,7}|#[xX][0-9a-fA-F]{1,6}|[A-Za-z][A-Za-z0-9]{1,31})\\x3B')))),
      'ARRAY<STRUCT<o: STRING, r: STRING>>')) AS decode_map
  FROM entity_rows
),
decoded AS (
  SELECT
    u.* EXCEPT (authors),
    CASE
      WHEN m.decode_map IS NOT NULL AND cardinality(m.decode_map) > 0
      THEN transform(u.authors, a -> named_struct(
        'given', a.given,
        'family', a.family,
        'name', a.name,
        'orcid', a.orcid,
        'affiliations', transform(a.affiliations, f -> named_struct(
          'name', COALESCE(try_element_at(m.decode_map, f.name), f.name),
          'department', f.department,
          'ror_id', f.ror_id)),
        'is_corresponding', a.is_corresponding,
        'author_key', a.author_key))
      ELSE u.authors
    END AS authors
  FROM unioned u
  LEFT JOIN entity_maps m
    ON  u.provenance = m.provenance
    AND u.native_id_namespace = m.native_id_namespace
    AND u.native_id = m.native_id
    AND u.work_id <=> m.work_id
),
-- oxjob #801: repair mojibake in affiliation strings for ALL provenances at map
-- time. The cheap RLIKE gate (a superset of the UDF's own detector: suspicious
-- lead/continuation pairs incl. cp1252 specials as continuations, plus the
-- latin2/cp1250 signature chars Ă/Ĺ) keeps the Python UDF off ~99.9% of rows.
-- The gate sits in a WHERE and the UDF in a projection above it (Spark hoists
-- UC Python UDFs out of CASE branches — see oxjob #801 PLAN gotchas), and the
-- UDF sees only the row's DISTINCT gate-matching names: the previous
-- whole-authors-array JSON round-trip OOMed the warehouse Python sandbox on
-- hyperauthorship works. Only affiliations[].name changes — author names are
-- deliberately NOT touched (they feed AER matching; oxjob #801).
-- Columns are enumerated to keep the table's column order unchanged.
gate_rows AS (
  SELECT provenance, native_id_namespace, native_id, work_id, authors
  FROM decoded
  WHERE EXISTS(authors, a -> EXISTS(a.affiliations, f -> f.name RLIKE '[\\u00C2-\\u00DF\\u00E0-\\u00EF][\\u0080-\\u00BF\\u20AC\\u201A\\u0192\\u201E\\u2026\\u2020\\u2021\\u02C6\\u2030\\u0160\\u2039\\u0152\\u017D\\u2018\\u2019\\u201C\\u201D\\u2022\\u2013\\u2014\\u02DC\\u2122\\u0161\\u203A\\u0153\\u017E\\u0178]|[\\u0102\\u0139]'))
),
repair_maps AS (
  SELECT
    provenance,
    native_id_namespace,
    native_id,
    work_id,
    map_from_entries(from_json(
      openalex.common.repair_mojibake_names_json(to_json(
        array_distinct(filter(
          flatten(transform(authors, a -> coalesce(transform(a.affiliations, f -> f.name), array()))),
          n -> n RLIKE '[\\u00C2-\\u00DF\\u00E0-\\u00EF][\\u0080-\\u00BF\\u20AC\\u201A\\u0192\\u201E\\u2026\\u2020\\u2021\\u02C6\\u2030\\u0160\\u2039\\u0152\\u017D\\u2018\\u2019\\u201C\\u201D\\u2022\\u2013\\u2014\\u02DC\\u2122\\u0161\\u203A\\u0153\\u017E\\u0178]|[\\u0102\\u0139]')))),
      'ARRAY<STRUCT<o: STRING, r: STRING>>')) AS repair_map
  FROM gate_rows
),
repaired AS (
  SELECT /*+ BROADCAST(om) */
    u.work_id,
    u.work_id_source,
    u.merge_key,
    u.key_lineage,
    u.provenance,
    u.native_id,
    u.native_id_namespace,
    u.title,
    u.normalized_title,
    CASE
      WHEN m.repair_map IS NOT NULL AND cardinality(m.repair_map) > 0
      THEN transform(u.authors, a -> named_struct(
        'given', a.given,
        'family', a.family,
        'name', a.name,
        'orcid', a.orcid,
        'affiliations', transform(a.affiliations, f -> named_struct(
          'name', COALESCE(try_element_at(m.repair_map, f.name), f.name),
          'department', f.department,
          'ror_id', f.ror_id)),
        'is_corresponding', a.is_corresponding,
        'author_key', a.author_key))
      ELSE u.authors
    END AS authors,
    u.ids,
    u.raw_type,
    u.type,
    u.version,
    u.license,
    u.language,
    u.language_classification,
    u.published_date,
    u.created_date,
    u.updated_date,
    u.issue,
    u.volume,
    u.first_page,
    u.last_page,
    u.is_retracted,
    u.abstract,
    u.source_name,
    u.publisher,
    u.funders,
    u.references,
    u.urls,
    -- oxjob #1934: an Elsevier Open Manuscript (a Crossref record Crossref.py labels an accepted manuscript, from its
    -- started am userlicense) with no PDF yet gets ScienceDirect's address for the manuscript, built from the PII in the
    -- same deposit (openalex.works.elsevier_open_manuscript_pii, the cell above). Before, ~286K bronze works served the
    -- DOI's paywalled article page as their best open copy. Landing page unchanged. Applied here, in the CTE read once,
    -- not in `live` (read four times). Revert: this CASE back to u.pdf_url and drop the om join below.
    CASE WHEN om.pii IS NOT NULL AND u.pdf_url IS NULL AND u.version = 'acceptedVersion'
              AND COALESCE(u.raw_type, '') <> 'posted-content'
         THEN concat('https://www.sciencedirect.com/science/article/am/pii/', om.pii)
         ELSE u.pdf_url END AS pdf_url,
    u.landing_page_url,
    u.pdf_s3_id,
    u.grobid_s3_id,
    u.mesh,
    u.is_oa,
    u.is_oa_source,
    u.referenced_works_count,
    u.referenced_works,
    u.abstract_inverted_index,
    u.authors_exist,
    u.affiliations_exist,
    u.is_corresponding_exists,
    u.best_doi,
    u.source_id,
    u.endpoint_id,
    u.ingested_at,
    u.openalex_created_dt
  FROM decoded u
  LEFT JOIN repair_maps m
    ON  u.provenance = m.provenance
    AND u.native_id_namespace = m.native_id_namespace
    AND u.native_id = m.native_id
    AND u.work_id <=> m.work_id
  LEFT JOIN openalex.works.elsevier_open_manuscript_pii om -- oxjob #1934 (one row per Elsevier DOI, ~299K)
    ON  u.provenance = 'crossref'
    AND u.native_id = om.native_id
)
SELECT
  u.* EXCEPT (openalex_created_dt),
  u.openalex_created_dt,
  CASE
    WHEN h.payload_hash = CASE
      -- Preserve the exact pre-cutover hash for rows with no acquisition lineage.
      WHEN u.endpoint_id IS NULL THEN xxhash64(to_json(struct(
        u.work_id, u.work_id_source, u.merge_key, u.key_lineage, u.title, u.normalized_title,
        u.authors, u.ids, u.raw_type, u.type, u.version, u.license, u.language, u.language_classification,
        u.published_date, u.created_date, u.updated_date, u.issue, u.volume, u.first_page, u.last_page,
        u.is_retracted, u.abstract, u.source_name, u.publisher, u.funders, u.references, u.urls,
        u.pdf_url, u.landing_page_url, u.pdf_s3_id, u.grobid_s3_id, u.mesh, u.is_oa, u.is_oa_source,
        u.referenced_works_count, u.referenced_works, u.abstract_inverted_index,
        u.authors_exist, u.affiliations_exist, u.is_corresponding_exists, u.best_doi, u.source_id
      )))
      ELSE xxhash64(to_json(struct(
        u.work_id, u.work_id_source, u.merge_key, u.key_lineage, u.title, u.normalized_title,
        u.authors, u.ids, u.raw_type, u.type, u.version, u.license, u.language, u.language_classification,
        u.published_date, u.created_date, u.updated_date, u.issue, u.volume, u.first_page, u.last_page,
        u.is_retracted, u.abstract, u.source_name, u.publisher, u.funders, u.references, u.urls,
        u.pdf_url, u.landing_page_url, u.pdf_s3_id, u.grobid_s3_id, u.mesh, u.is_oa, u.is_oa_source,
        u.referenced_works_count, u.referenced_works, u.abstract_inverted_index,
        u.authors_exist, u.affiliations_exist, u.is_corresponding_exists, u.best_doi, u.source_id,
        u.endpoint_id
      )))
    END THEN h.openalex_updated_dt
    ELSE current_timestamp()
  END AS openalex_updated_dt
FROM repaired u
LEFT JOIN openalex.works.locations_mapped_hash h
  ON  u.provenance = h.provenance
  AND u.native_id_namespace = h.native_id_namespace
  AND u.native_id = h.native_id
  AND u.work_id <=> h.work_id

## Refresh the hash table from the rebuilt output
Rows with NULL acquisition lineage keep the byte-identical legacy payload formula;
non-NULL `endpoint_id` is hashed so future lineage corrections bump the timestamp.

In [0]:
CREATE OR REPLACE TABLE openalex.works.locations_mapped_hash
CLUSTER BY (provenance, native_id_namespace, native_id)
AS
SELECT provenance, native_id_namespace, native_id, work_id,
       CASE
         WHEN endpoint_id IS NULL THEN xxhash64(to_json(struct(
           work_id, work_id_source, merge_key, key_lineage, title, normalized_title,
           authors, ids, raw_type, type, version, license, language, language_classification,
           published_date, created_date, updated_date, issue, volume, first_page, last_page,
           is_retracted, abstract, source_name, publisher, funders, references, urls,
           pdf_url, landing_page_url, pdf_s3_id, grobid_s3_id, mesh, is_oa, is_oa_source,
           referenced_works_count, referenced_works, abstract_inverted_index,
           authors_exist, affiliations_exist, is_corresponding_exists, best_doi, source_id
         )))
         ELSE xxhash64(to_json(struct(
           work_id, work_id_source, merge_key, key_lineage, title, normalized_title,
           authors, ids, raw_type, type, version, license, language, language_classification,
           published_date, created_date, updated_date, issue, volume, first_page, last_page,
           is_retracted, abstract, source_name, publisher, funders, references, urls,
           pdf_url, landing_page_url, pdf_s3_id, grobid_s3_id, mesh, is_oa, is_oa_source,
           referenced_works_count, referenced_works, abstract_inverted_index,
           authors_exist, affiliations_exist, is_corresponding_exists, best_doi, source_id,
           endpoint_id
         )))
       END AS payload_hash,
       openalex_updated_dt
FROM openalex.works.locations_mapped
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY provenance, native_id_namespace, native_id, work_id
  ORDER BY openalex_updated_dt DESC NULLS LAST) = 1

In [0]:
SELECT format_number(COUNT(*), 0) AS row_count,
       format_number(COUNT(DISTINCT work_id), 0) AS distinct_works
FROM openalex.works.locations_mapped